In [ ]:
# check langchain version if not uncomment below pip command to install the correct version
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)
# !pip install langchain==0.3.27 langchain-openai==0.3.33 langchain-community==0.3.24


0.1.20
0.0.38


In [2]:
import os
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv
# Load environment variables
load_dotenv()

import warnings
warnings.filterwarnings('ignore')

In [3]:
from langchain.prompts import ChatPromptTemplate
from langchain.agents import AgentExecutor, create_react_agent
from langchain.tools import tool

In [4]:
# The LLM (Agent Brain)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [5]:
# 2️⃣ Define Tool Using @tool Decorator
@tool
def multiply_numbers(query: str) -> str:
    """
    Multiply two numbers found in a text input.
    Input: string containing two numeric values.
    Output: numeric multiplication result.
    """
    try:
        nums = [float(x) for x in query.split() if x.replace('.', '', 1).isdigit()]
        return str(nums[0] * nums[1])
    except:
        return "Couldn't extract numeric values."



In [6]:
# Tools list
tools = [multiply_numbers]

In [7]:
# Tool-Calling Prompt 
prompt = ChatPromptTemplate.from_template("""
You are a helpful reasoning AI agent.

Follow this decision process:
1. Understand the user request.
2. Decide whether a tool is needed.
3. If yes → call the correct tool.
4. If no → answer directly.
5. Reflect before final output.

User Query: {input},
("placeholder", "{agent_scratchpad}")
""")

In [8]:
# Create Agent With Tool Calling
from langchain.agents import AgentExecutor, create_tool_calling_agent

agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [9]:
# Agent Runtime using AgentExecutor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=1 )

In [10]:
# Test agent
result = agent_executor.invoke({"input": "Multiply 15 and 4"})
print("\nFINAL OUTPUT:", result["output"])



> Entering new AgentExecutor chain...

Invoking: `multiply_numbers` with `{'query': 'Multiply 15 and 4'}`


60.0

> Finished chain.

FINAL OUTPUT: Agent stopped due to max iterations.


The agent doesn't just answer.
It follows a cognitive workflow:
Interpret the query
Reason about whether a tool is needed
Call the tool (if required)
Observe the result
Reflect
Produce the final answer

Each cycle (Thought → Action → Observation → Reflection) counts as an iteration.
LangChain sets a default max_iterations limit (usually 15 or 20 depending on the version) to prevent: use max_iterations=1 in AgentExecutor


##similar to above create industry specific simple agent using tool calling

In [11]:
# Define a new tool 
@tool
def industry_insights(query: str) -> str:

    """
    Provide insights on a given industry topic.
    Input: string containing industry-related query.
    Output: string with insights.
    """
    insights = {
        "renewable energy": "Renewable energy is growing rapidly due to environmental concerns and technological advancements.",
        "artificial intelligence": "AI is transforming industries by automating tasks and providing data-driven insights.",
        "blockchain": "Blockchain technology offers decentralized solutions for secure transactions and data integrity."
    }
    return insights.get(query.lower(), "No insights available for this topic.") 


In [12]:
# New Tools list
industry_tools = [industry_insights]

# Create Industry Agent With Tool Calling
industry_agent = create_tool_calling_agent(
    llm=llm,
    tools=industry_tools,
    prompt=prompt  # using same prompt structure as above
)


In [13]:


# Industry Agent Runtime using AgentExecutor
industry_agent_executor = AgentExecutor(agent=industry_agent, tools=industry_tools, verbose=True,
                                        max_iterations=1 )  
# Test industry agent
industry_result = industry_agent_executor.invoke({"input": "Provide insights on renewable energy"})
print("\nFINAL OUTPUT:", industry_result["output"])



> Entering new AgentExecutor chain...

Invoking: `industry_insights` with `{'query': 'renewable energy'}`


Renewable energy is growing rapidly due to environmental concerns and technological advancements.

> Finished chain.

FINAL OUTPUT: Agent stopped due to max iterations.
